In [ ]:
!pip install -q google-generativeai pandas tqdm


In [ ]:
import pandas as pd
import json
import time

from tqdm import tqdm

from openai import OpenAI

In [ ]:
from openai import OpenAI

API_KEY = "some_api_key"

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ]
)

print(response.choices[0].message.content)

Hello, how can I assist you today?


In [ ]:
print(API_KEY[:10])

gsk_uQaasW


In [ ]:
df = pd.read_csv("/content/text_asr.csv")

df.head()

,complaint_id,asr_attempt_id,text_raw,asr_lang,is_primary,transcript_ts,asr_model_version
0,PGR202500001,1,"Namaste, for the past three days. We are exper...",mr,True,2025-02-14 09:18:54,asr_mr_v3
1,PGR202500002,1,"Ji, for the past three days. I am reporting th...",mr,True,2025-05-28 02:15:39,asr_mr_v3
2,PGR202500003,1,"Good evening, for 48 hours. We are experiencin...",hi,True,2025-05-10 04:28:36,asr_hi_v3
3,PGR202500004,1,"Good evening, since last night. an old buildin...",mr,True,2025-04-07 19:58:15,asr_mr_v3
4,PGR202500005,1,"Hi, for the past three days. We are experienci...",hi,True,2025-04-06 22:51:29,asr_hi_v3


In [ ]:
if "is_primary" in df.columns:
    df = df[df["is_primary"] == True]

df = df[
    [
        "complaint_id",
        "text_raw"
    ]
]

df = df.drop_duplicates("complaint_id")

df.reset_index(drop=True, inplace=True)

print(df.shape)

(900, 2)


In [ ]:
SYSTEM_PROMPT = """
You are an experienced municipal grievance officer.

You are given ONE civic complaint.

Your task is to assess its severity.

Evaluate ONLY using the complaint text.

Do not assume information that is not explicitly mentioned.

Assign:

1. severity_score (0-10)

2. severity_label

Possible labels:

LOW
MEDIUM
HIGH
CRITICAL

Guidelines

LOW:
Minor inconvenience.
No immediate risk.

MEDIUM:
Moderate disruption affecting a few citizens.

HIGH:
Serious infrastructure damage or significant public inconvenience.

CRITICAL:
Immediate danger to life, emergency services, fire, electrocution,
building collapse, severe flooding, major accidents, etc.

Also provide:

confidence (0-1)

reason (one sentence)

Return ONLY JSON.

Format:

{
"severity_score":8.5,
"severity_label":"HIGH",
"confidence":0.93,
"reason":"Tree fallen across main road causing traffic blockage."
}
"""

In [ ]:
def label_complaint(text):

    response = client.chat.completions.create(
    model="llama-3.1-8b-instant",



        temperature=0,

        response_format={"type": "json_object"},

        messages=[

            {
                "role":"system",
                "content":SYSTEM_PROMPT
            },

            {
                "role":"user",
                "content":text
            }

        ]
    )

    return json.loads(
        response.choices[0].message.content
    )

In [ ]:
sample = df.iloc[0]["text_raw"]

print(sample)

print()

print(label_complaint(sample))

Namaste, for the past three days. We are experiencing garbage pickup has been missed repeatedly. we also tried the IVR earlier. loker madat kara. krupaya sahayya kara.

{'severity_score': 4, 'severity_label': 'MEDIUM', 'confidence': 0.8, 'reason': 'Repeated missed garbage pickup causing moderate inconvenience to residents.'}


In [ ]:
import pandas as pd
from tqdm import tqdm
import time
import json

results = []

In [ ]:
results = []

SAVE_EVERY = 25

for index, row in tqdm(df.iterrows(), total=len(df)):

    complaint = row["text_raw"]

    try:

        output = label_complaint(complaint)

        results.append({

            "complaint_id": row["complaint_id"],
            "complaint": complaint,
            "severity_score": output["severity_score"],
            "severity_label": output["severity_label"],
            "confidence": output["confidence"],
            "reason": output["reason"]

        })

    except Exception as e:

        print(e)

        results.append({

            "complaint_id": row["complaint_id"],
            "complaint": complaint,
            "severity_score": None,
            "severity_label": None,
            "confidence": None,
            "reason": str(e)

        })

    if (index + 1) % SAVE_EVERY == 0:

        pd.DataFrame(results).to_csv(
            "severity_llm_labels.csv",
            index=False
        )

        print(f"Saved till row {index+1}")

  3%|▎         | 25/900 [00:23<42:07,  2.89s/it]

Saved till row 25


  6%|▌         | 50/900 [01:43<46:26,  3.28s/it]

Saved till row 50


  8%|▊         | 75/900 [03:04<44:50,  3.26s/it]

...

Saved till row 850


 97%|█████████▋| 875/900 [46:12<01:20,  3.23s/it]

Saved till row 875


100%|██████████| 900/900 [47:32<00:00,  3.17s/it]

Saved till row 900


In [ ]:
df.to_excel("text_asr_enriched.xlsx", index=False)

In [ ]:
print(df.columns)

Index(['complaint_id', 'text_raw'], dtype='object')


In [ ]:
results[0]

{'complaint_id': 'PGR202500001',
 'complaint': 'Namaste, for the past three days. We are experiencing garbage pickup has been missed repeatedly. we also tried the IVR earlier. loker madat kara. krupaya sahayya kara.',
 'severity_score': 4,
 'severity_label': 'MEDIUM',
 'confidence': 0.8,
 'reason': 'Repeated missed garbage pickup causing moderate inconvenience to residents.'}

In [ ]:
import pandas as pd

# Convert results to DataFrame
results_df = pd.DataFrame(results)

print(results_df.head())

   complaint_id                                          complaint  \
0  PGR202500001  Namaste, for the past three days. We are exper...   
1  PGR202500002  Ji, for the past three days. I am reporting th...   
2  PGR202500003  Good evening, for 48 hours. We are experiencin...   
3  PGR202500004  Good evening, since last night. an old buildin...   
4  PGR202500005  Hi, for the past three days. We are experienci...   

   severity_score severity_label  confidence  \
0             4.0         MEDIUM        0.80   
1             4.0         MEDIUM        0.80   
2             6.0         MEDIUM        0.85   
3             8.0           HIGH        0.85   
4             8.0           HIGH        0.95   

                                              reason  
0  Repeated missed garbage pickup causing moderat...  
1  Repeated missed garbage pickup causing inconve...  
2  Residents are experiencing a 48-hour water out...  
3  Cracked wall in an old building poses a risk o...  
4  Septic tank 

In [ ]:
enriched_df = df.merge(
    results_df[
        [
            "complaint_id",
            "severity_score",
            "severity_label",
            "confidence",
            "reason"
        ]
    ],
    on="complaint_id",
    how="left"
)

In [ ]:
print(enriched_df.head())
print(enriched_df.columns)

   complaint_id                                           text_raw  \
0  PGR202500001  Namaste, for the past three days. We are exper...   
1  PGR202500002  Ji, for the past three days. I am reporting th...   
2  PGR202500003  Good evening, for 48 hours. We are experiencin...   
3  PGR202500004  Good evening, since last night. an old buildin...   
4  PGR202500005  Hi, for the past three days. We are experienci...   

   severity_score severity_label  confidence  \
0             4.0         MEDIUM        0.80   
1             4.0         MEDIUM        0.80   
2             6.0         MEDIUM        0.85   
3             8.0           HIGH        0.85   
4             8.0           HIGH        0.95   

                                              reason  
0  Repeated missed garbage pickup causing moderat...  
1  Repeated missed garbage pickup causing inconve...  
2  Residents are experiencing a 48-hour water out...  
3  Cracked wall in an old building poses a risk o...  
4  Septic tank 

In [ ]:
enriched_df.to_excel("text_asr_enriched_82percent.xlsx", index=False)

from google.colab import files
files.download("text_asr_enriched_82percent.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install sentence-transformers catboost scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_excel("text_asr_enriched.xlsx")

df.head()

,complaint_id,text_raw,severity_score,severity_label,confidence,reason
0,PGR202500001,"Namaste, for the past three days. We are exper...",4.0,MEDIUM,0.80,Repeated missed garbage pickup causing moderat...
1,PGR202500002,"Ji, for the past three days. I am reporting th...",4.0,MEDIUM,0.80,Repeated missed garbage pickup causing inconve...
2,PGR202500003,"Good evening, for 48 hours. We are experiencin...",6.0,MEDIUM,0.85,Residents are experiencing a 48-hour water out...
3,PGR202500004,"Good evening, since last night. an old buildin...",8.0,HIGH,0.85,Cracked wall in an old building poses a risk o...
4,PGR202500005,"Hi, for the past three days. We are experienci...",8.0,HIGH,0.95,"Septic tank overflow onto the street, causing ..."


In [ ]:
df = df.dropna(subset=["severity_score"])

print(len(df))

744


In [ ]:
X = df["text_raw"].astype(str)

y = df["severity_score"].astype(float)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

X_embeddings = model.encode(
    X.tolist(),
    show_progress_bar=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_embeddings,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from catboost import CatBoostRegressor

cat = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=100,
    random_seed=42
)

cat.fit(
    X_train,
    y_train
)

0:	learn: 2.3516819	total: 824ms	remaining: 6m 51s
100:	learn: 0.8197675	total: 23s	remaining: 1m 30s
200:	learn: 0.4689540	total: 34.5s	remaining: 51.3s
300:	learn: 0.3102033	total: 45.6s	remaining: 30.1s
400:	learn: 0.2270210	total: 56.6s	remaining: 14s
499:	learn: 0.1583678	total: 1m 6s	remaining: 0us


CatBoostRegressor(depth=6, iterations=500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

pred = cat.predict(X_test)

print("MAE :", mean_absolute_error(y_test,pred))
print("RMSE:", mean_squared_error(y_test,pred)**0.5)
print("R² :", r2_score(y_test,pred))

MAE : 1.0168905113329763
RMSE: 1.3687507834303216
R² : 0.7320163740849837


In [ ]:
import joblib

cat.save_model("severity_model.cbm")

joblib.dump(model,"sentence_transformer.pkl")

['sentence_transformer.pkl']